# Calibration and operating-threshold selection (M0)

This notebook consumes the selected P95 and P90 model specifications from `05_model_training_tuning.ipynb`. It fits each selected base model on the frozen `model_development` block, fits a Platt sigmoid calibrator on `calibration`, and selects one operating threshold on `threshold_selection`. The outer test rows and labels are excluded throughout.

The threshold-selection period is used to select the operating point, so its threshold metrics are selection results rather than an independent estimate of future performance. Notebook 08B evaluates these primary full-M0 models on the outer test; Notebook 08 independently fits the weather-paired ablation for RQ3 after the model, calibrators, and thresholds are locked.

## 1. Imports, paths, and reproducible policy

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score,
                             log_loss, precision_score, recall_score)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "data" / "processed").exists():
    raise RuntimeError("Could not locate project root containing data/processed")

DATA_DIR = ROOT / "data" / "processed"
TABLE_DIR = ROOT / "reports" / "tables" / "06_calibration_evaluation"
FIGURE_DIR = ROOT / "reports" / "figures" / "06_calibration_evaluation"
MODEL_DIR = ROOT / "models" / "06_calibrated_m0"
for directory in (TABLE_DIR, FIGURE_DIR, MODEL_DIR): directory.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_DIR / "04_master_split.csv"
SELECTED_PATH = ROOT / "reports" / "tables" / "05_model_training_tuning" / "05_selected_models.json"
SPLIT_MANIFEST_PATH = ROOT / "reports" / "tables" / "04_split_design" / "04_split_design_manifest.json"
TARGET_CONFIG_PATH = ROOT / "reports" / "tables" / "03a_target_design_experiments" / "03a_selected_target_config.json"
FEATURE_QUALITY_CONFIG_PATH = ROOT / "reports" / "tables" / "03b_historical_feature_quality_experiments" / "03b_selected_feature_quality_config.json"
RANDOM_STATE = 42
THRESHOLD_GRID = np.round(np.arange(0.01, 1.00, 0.01), 2)
INTERNAL_STAGES = ("model_development", "calibration", "threshold_selection")
TARGETS = {"P95": DATA_DIR / "05_model_ready_m0_p95.csv",
           "P90": DATA_DIR / "05_model_ready_m0_p90.csv"}

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(8*1024*1024),b""): h.update(block)
    return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))

def numeric_x(frame, features):
    return frame[features].apply(lambda c: pd.to_numeric(c, errors="coerce").astype("float64"))


## 2. Verify handoffs and read only the three internal development blocks

The master split is the source of row membership. The loader first scans only `row_id` from each model-ready CSV, then parses feature and target values only for rows assigned to `model_development`, `calibration`, or `threshold_selection`. Outer-test feature and target rows are skipped before model data are parsed.

In [ ]:
if not MASTER_PATH.exists() or not SELECTED_PATH.exists() or not SPLIT_MANIFEST_PATH.exists():
    raise FileNotFoundError("Run Notebooks 04 and 05 before this notebook")
master = pd.read_csv(MASTER_PATH, dtype={"row_id":"string", "household_id":"string"})
master["target_date"] = pd.to_datetime(master["target_date"], errors="raise")
if master.row_id.isna().any() or master.row_id.duplicated().any(): raise ValueError("Master split row_id must be unique and non-null")
if not set(master.internal_split.dropna().unique()).issuperset(INTERNAL_STAGES): raise ValueError("Expected internal split blocks are missing")
if master.loc[master.outer_split.eq("test"), "internal_split"].isin(INTERNAL_STAGES).any(): raise ValueError("Outer-test rows must not have an internal development assignment")

selected_models = read_json(SELECTED_PATH)
if set(selected_models) != {"P95", "P90"}: raise ValueError("Selected-model handoff must contain P95 and P90")
split_manifest = read_json(SPLIT_MANIFEST_PATH)

def read_internal_only(path, expected_ids):
    # Scan row IDs only; use physical CSV line positions to skip all rows outside the internal blocks.
    row_ids = pd.read_csv(path, usecols=["row_id"], dtype={"row_id":"string"}).row_id
    matched = np.flatnonzero(row_ids.isin(expected_ids).to_numpy())
    found = set(row_ids.iloc[matched].astype(str))
    if found != expected_ids:
        raise ValueError(f"Input/master row_id mismatch for {path.name}: missing={len(expected_ids-found)}, extra={len(found-expected_ids)}")
    keep_positions = set(matched.tolist())
    frame = pd.read_csv(path, dtype={"row_id":"string", "household_id":"string"},
                        skiprows=lambda line: line > 0 and (line-1) not in keep_positions)
    if set(frame.row_id.astype(str)) != expected_ids or frame.row_id.duplicated().any():
        raise ValueError(f"Selective loader returned unexpected IDs for {path.name}")
    return frame

expected_ids = set(master.loc[master.internal_split.isin(INTERNAL_STAGES), "row_id"].astype(str))
if not expected_ids: raise ValueError("No internal development rows found")
frames = {}
for target, path in TARGETS.items():
    frame = read_internal_only(path, expected_ids)
    frame["measurement_date"] = pd.to_datetime(frame["measurement_date"], errors="raise")
    assignment = master[["row_id", "outer_split", "internal_split", "target_date"]]
    frame = frame.merge(assignment, on="row_id", how="left", validate="one_to_one")
    if frame.internal_split.isna().any() or frame.outer_split.eq("test").any(): raise ValueError(f"{target}: outer-test rows or unmatched rows were loaded")
    if not frame.measurement_date.eq(frame.target_date).all(): raise ValueError(f"{target}: date mismatch against master split")
    features = selected_models[target]["feature_list"]
    missing = sorted(set(features + ["target"]) - set(frame.columns))
    if missing: raise ValueError(f"{target}: missing required columns {missing}")
    if frame.target.isna().any() or not set(frame.target.astype(int).unique()).issubset({0,1}): raise ValueError(f"{target}: invalid target labels")
    frames[target] = frame

period_rows=[]
for target, frame in frames.items():
    for stage in INTERNAL_STAGES:
        part=frame.loc[frame.internal_split.eq(stage)]
        period_rows.append({"target":target,"internal_split":stage,"rows":len(part),"positive_events":int(part.target.sum()),
                            "prevalence":float(part.target.mean()),"start_date":part.measurement_date.min(),"end_date":part.measurement_date.max(),
                            "unique_dates":int(part.measurement_date.nunique())})
period_summary=pd.DataFrame(period_rows)
display(period_summary)
period_summary.to_csv(TABLE_DIR/"06_internal_period_summary.csv",index=False)


## 3. Fit the selected base estimators on `model_development` only

The selected estimator family and parameters come directly from Notebook 05. Preprocessing is fitted with the estimator on the model-development rows. XGBoost class weighting is derived from model-development labels only.

In [ ]:
def build_estimator(target, family, params, y_fit):
    params=dict(params)
    if family == "XGBoost":
        weight_mode=params.pop("scale_pos_weight_mode", "one")
        ratio=float((y_fit==0).sum()/max((y_fit==1).sum(),1))
        multipliers={"one":0.0,"sqrt_ratio":0.5,"half_ratio":0.5,"full_ratio":1.0}
        if weight_mode == "one": scale=1.0
        elif weight_mode == "sqrt_ratio": scale=ratio**0.5
        elif weight_mode == "half_ratio": scale=ratio*0.5
        elif weight_mode == "full_ratio": scale=ratio
        else: raise ValueError(f"Unknown scale_pos_weight_mode={weight_mode}")
        model=XGBClassifier(**params, scale_pos_weight=scale, eval_metric="logloss", tree_method="hist",
                            n_jobs=2, random_state=RANDOM_STATE, verbosity=0)
        return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("model",model)])
    if family == "Random Forest":
        model=RandomForestClassifier(**params,n_jobs=2,random_state=RANDOM_STATE)
        return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("model",model)])
    if family == "Logistic Regression":
        model=LogisticRegression(**params,solver="lbfgs",max_iter=2000,random_state=RANDOM_STATE)
        return Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),("model",model)])
    if family == "CatBoost":
        model=CatBoostClassifier(**params,loss_function="Logloss",random_seed=RANDOM_STATE,verbose=False,
                                 allow_writing_files=False,thread_count=2)
        return model
    raise ValueError(f"Unsupported selected model family: {family}")

def probability_logit(probability):
    p=np.clip(np.asarray(probability,dtype=float),1e-6,1-1e-6)
    return np.log(p/(1-p)).reshape(-1,1)

base_models={}; raw_probs={}; calibrators={}; calibrated_probs={}; threshold_rows=[]
for target, frame in frames.items():
    spec=selected_models[target]; features=spec["feature_list"]; family=spec["model_family"]
    fit=frame.loc[frame.internal_split.eq("model_development")].copy()
    cal=frame.loc[frame.internal_split.eq("calibration")].copy()
    threshold=frame.loc[frame.internal_split.eq("threshold_selection")].copy()
    if fit.empty or cal.empty or threshold.empty: raise ValueError(f"{target}: one or more internal blocks are empty")
    y_fit=fit.target.astype("int8").to_numpy()
    estimator=build_estimator(target,family,spec["parameters"],y_fit)
    X_fit=numeric_x(fit,features)
    estimator.fit(X_fit,y_fit)
    p_cal=estimator.predict_proba(numeric_x(cal,features))[:,1]
    p_threshold=estimator.predict_proba(numeric_x(threshold,features))[:,1]
    calibrator=LogisticRegression(C=1.0,solver="lbfgs",max_iter=2000,random_state=RANDOM_STATE)
    calibrator.fit(probability_logit(p_cal),cal.target.astype("int8").to_numpy())
    p_calibrated=calibrator.predict_proba(probability_logit(p_threshold))[:,1]
    base_models[target]=estimator; raw_probs[target]={"calibration":p_cal,"threshold_selection":p_threshold}
    calibrators[target]=calibrator; calibrated_probs[target]={"calibration":calibrator.predict_proba(probability_logit(p_cal))[:,1],
                                                                  "threshold_selection":p_calibrated}
    print(f"{target}: selected {family}; fitted {len(fit):,} rows; calibration {len(cal):,}; threshold-selection {len(threshold):,}")


## 4. Calibrate probabilities and choose thresholds on the separate later block

Platt scaling is fitted only on `calibration`. The operating threshold maximizes F1 on `threshold_selection`; ties prefer higher precision and then a higher threshold. Threshold-selection metrics are reported as selection diagnostics.

In [ ]:
def threshold_search(y_true, probability, target):
    rows=[]
    for threshold in THRESHOLD_GRID:
        pred=(probability>=threshold).astype("int8")
        rows.append({"target":target,"threshold":float(threshold),"precision":precision_score(y_true,pred,zero_division=0),
                     "recall":recall_score(y_true,pred,zero_division=0),"f1":f1_score(y_true,pred,zero_division=0)})
    result=pd.DataFrame(rows)
    chosen=result.sort_values(["f1","precision","threshold"],ascending=[False,False,False],kind="stable").iloc[0]
    return result,chosen

threshold_curves=[]; chosen_thresholds={}; quality_rows=[]
for target,frame in frames.items():
    for stage in ("calibration","threshold_selection"):
        part=frame.loc[frame.internal_split.eq(stage)]
        y=part.target.astype("int8").to_numpy()
        for method,probs in (("raw",raw_probs[target][stage]),("platt_calibrated",calibrated_probs[target][stage])):
            quality_rows.append({"target":target,"stage":stage,"probability":"raw" if method=="raw" else "platt_calibrated",
                                 "rows":len(y),"positive_events":int(y.sum()),"brier_score":brier_score_loss(y,probs),
                                 "log_loss":log_loss(y,probs,labels=[0,1]),
                                 "interpretation":"in-sample calibrator fit diagnostic" if stage=="calibration" else "held apart from calibrator fit; threshold-selection period"})
    threshold_part=frame.loc[frame.internal_split.eq("threshold_selection")]
    curve,chosen=threshold_search(threshold_part.target.astype("int8").to_numpy(),calibrated_probs[target]["threshold_selection"],target)
    threshold_curves.append(curve); chosen_thresholds[target]=float(chosen.threshold)
    print(f"{target}: threshold={chosen.threshold:.2f}, F1={chosen.f1:.3f}, precision={chosen.precision:.3f}, recall={chosen.recall:.3f}")

quality=pd.DataFrame(quality_rows)
threshold_curves=pd.concat(threshold_curves,ignore_index=True)
threshold_table=pd.DataFrame([{"target":t,"model_family":selected_models[t]["model_family"],
                              "threshold":chosen_thresholds[t],"threshold_rule":"maximize F1; ties by precision then higher threshold",
                              "threshold_selection_rows":int((frames[t].internal_split=="threshold_selection").sum()),
                              "threshold_selection_events":int(frames[t].loc[frames[t].internal_split.eq("threshold_selection"),"target"].sum())}
                             for t in frames])
display(threshold_table); display(quality)
threshold_table.to_csv(TABLE_DIR/"06_locked_thresholds_m0.csv",index=False)
threshold_curves.to_csv(TABLE_DIR/"06_threshold_search_development.csv",index=False)
quality.to_csv(TABLE_DIR/"06_probability_quality_development.csv",index=False)


## 5. Development-only diagnostics

Reliability diagrams and class-conditional probability histograms use `threshold_selection`, which did not fit the calibrator. Threshold curves use the same block because it defines the operating threshold; these curves are not independent performance estimates.

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(12,9))
for ax,target in zip(axes[0],("P95","P90")):
    frame=frames[target]; part=frame.loc[frame.internal_split.eq("threshold_selection")]; y=part.target.astype("int8").to_numpy()
    for label,probs in (("Raw",raw_probs[target]["threshold_selection"]),("Platt calibrated",calibrated_probs[target]["threshold_selection"])):
        frac,mean=calibration_curve(y,probs,n_bins=10,strategy="quantile")
        ax.plot(mean,frac,marker="o",label=label)
    ax.plot([0,1],[0,1],"--",color="gray"); ax.set(title=f"{target}: reliability on threshold-selection period",xlabel="Mean predicted probability",ylabel="Observed event rate"); ax.legend()
for ax,target in zip(axes[1],("P95","P90")):
    frame=frames[target]; y=frame.loc[frame.internal_split.eq("threshold_selection"),"target"].astype("int8").to_numpy()
    prob=calibrated_probs[target]["threshold_selection"]
    ax.hist(prob[y==0],bins=30,density=True,alpha=.55,label="Target=0")
    ax.hist(prob[y==1],bins=30,density=True,alpha=.55,label="Target=1")
    ax.axvline(chosen_thresholds[target],color="black",linestyle="--",label=f"Selected threshold {chosen_thresholds[target]:.2f}")
    ax.set(title=f"{target}: calibrated probabilities by class",xlabel="Calibrated probability",ylabel="Density"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR/"06_calibration_and_probability_diagnostics.png",dpi=160,bbox_inches="tight"); plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,target in zip(axes,("P95","P90")):
    curve=threshold_curves.loc[threshold_curves.target.eq(target)]
    for metric in ("precision","recall","f1"): ax.plot(curve.threshold,curve[metric],label=metric.title())
    ax.axvline(chosen_thresholds[target],color="black",linestyle="--",label=f"Selected {chosen_thresholds[target]:.2f}")
    ax.set(title=f"{target}: threshold-selection curves",xlabel="Threshold",ylabel="Score",ylim=(0,1)); ax.legend()
fig.tight_layout(); fig.savefig(FIGURE_DIR/"06_threshold_selection_curves.png",dpi=160,bbox_inches="tight"); plt.show()


## 6. Save frozen model bundles and validate the handoff

Each bundle contains the selected base estimator, the separately fitted Platt calibrator, its feature list, and the selected threshold. The outer test remains untouched for Notebook 08.

In [ ]:
target_config=read_json(TARGET_CONFIG_PATH); feature_quality_config=read_json(FEATURE_QUALITY_CONFIG_PATH)
if target_config.get("status")!="frozen_primary_research_policy" or feature_quality_config.get("status")!="frozen_feature_quality_policy": raise RuntimeError("Frozen target/feature-quality policy provenance is required")
for target in frames:
    bundle={"target":target,"model_role":"primary_full_m0","research_questions":["RQ2","RQ4"],
            "target_configuration_id":target_config["configuration_id"],"feature_quality_configuration_id":feature_quality_config["configuration_id"],"training_population":"full_m0_model_development",
            "model_family":selected_models[target]["model_family"],
            "selected_parameters":selected_models[target]["parameters"],"features":selected_models[target]["feature_list"],
            "base_estimator":base_models[target],"platt_calibrator":calibrators[target],
            "threshold":chosen_thresholds[target],"threshold_rule":"maximize F1 on threshold_selection; ties by precision then higher threshold",
            "fit_split":"model_development","calibration_split":"calibration","threshold_split":"threshold_selection",
            "outer_test_used_for_fit_calibration_or_threshold":False}
    joblib.dump(bundle,MODEL_DIR/f"{target.lower()}_calibrated_m0.joblib")

checks=[]
def check(name,passed,detail): checks.append({"check":name,"passed":bool(passed),"detail":str(detail)})
check("both_selected_targets_present",set(frames)=={"P95","P90"},sorted(frames))
check("only_internal_development_ids_loaded",all(set(f.row_id.astype(str))==expected_ids for f in frames.values()),len(expected_ids))
check("outer_test_rows_not_loaded",all(not f.outer_split.eq("test").any() for f in frames.values()),"Only three internal_split blocks were parsed")
check("all_three_internal_blocks_present",all(set(f.internal_split.unique())==set(INTERNAL_STAGES) for f in frames.values()),INTERNAL_STAGES)
check("calibrator_fitted_on_calibration_only",all(len(raw_probs[t]["calibration"])==(frames[t].internal_split=="calibration").sum() for t in frames),"Platt inputs came from calibration predictions")
check("threshold_chosen_on_threshold_selection_only",all(0.01<=v<=0.99 for v in chosen_thresholds.values()),chosen_thresholds)
check("model_bundles_saved",all((MODEL_DIR/f"{t.lower()}_calibrated_m0.joblib").exists() for t in frames),MODEL_DIR)
roundtrip_ok=True; roundtrip_details={}
for target, frame in frames.items():
    bundle=joblib.load(MODEL_DIR/f"{target.lower()}_calibrated_m0.joblib")
    sample=frame.loc[frame.internal_split.eq("threshold_selection")].head(12)
    raw=bundle["base_estimator"].predict_proba(numeric_x(sample,bundle["features"]))[:,1]
    calibrated=bundle["platt_calibrator"].predict_proba(probability_logit(raw))[:,1]
    valid=(bundle["target"]==target and len(calibrated)==len(sample) and np.isfinite(calibrated).all() and ((calibrated>=0)&(calibrated<=1)).all())
    roundtrip_ok=roundtrip_ok and bool(valid); roundtrip_details[target]={"rows_checked":len(sample),"probabilities_finite_and_bounded":bool(valid)}
check("saved_bundles_reload_and_predict",roundtrip_ok,roundtrip_details)
validation=pd.DataFrame(checks); display(validation)
validation.to_csv(TABLE_DIR/"06_final_validation.csv",index=False)
if not validation.passed.all(): raise AssertionError("One or more Notebook 06 validation checks failed")
manifest={"created_utc":datetime.now(timezone.utc).isoformat(),"stage":"06_calibration_and_threshold_design",
          "selected_models_sha256":sha256(SELECTED_PATH),"master_split_sha256":sha256(MASTER_PATH),
          "outer_test_labels_loaded":False,"outer_test_scored":False,"outer_test_used_for_fit_calibration_or_threshold":False,
          "base_fit_split":"model_development","calibrator_fit_split":"calibration","threshold_selection_split":"threshold_selection",
          "threshold_rule":"maximize F1 on threshold_selection; ties by precision then higher threshold",
          "primary_model_role":"primary_full_m0","target_configuration_id":target_config["configuration_id"],"feature_quality_configuration_id":feature_quality_config["configuration_id"],
          "selected_models":{t:{"model_role":"primary_full_m0","research_questions":["RQ2","RQ4"],"model_family":selected_models[t]["model_family"],"threshold":chosen_thresholds[t],
                               "bundle":str(MODEL_DIR/f"{t.lower()}_calibrated_m0.joblib")} for t in frames},
          "validation_passed":bool(validation.passed.all()),"validation_checks":checks,
          "interpretation":"Calibration-period quality is in-sample for Platt fitting. Threshold-selection quality and F1-based operating points are selection results, not independent future performance estimates."}
(TABLE_DIR/"06_calibration_and_threshold_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("FINAL MODEL LOCKED — base models, calibrators, and operating thresholds are frozen for the outer-test evaluation.")


## Final status

**FINAL MODEL LOCKED.** No further modelling decisions are allowed after these artifacts are produced. Notebook 08 can now evaluate the frozen model bundles and thresholds once on the reserved outer test.

The calibration diagnostics on `calibration` are in-sample because that block fits Platt scaling. The threshold-selection results are also selection results because that block selects the F1 operating point. Neither should be interpreted as an unbiased outer-test performance estimate.